# Federal Reserve Macro Insights MVP

Minimum-viable pipeline using only free/open-source components: scrape Federal Reserve PDFs, build local retrieval, and generate a structured macro view with citations via a local LLM.

## 0) Install dependencies

In [1]:
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -q requests beautifulsoup4 pypdf sentence-transformers faiss-cpu numpy pandas tqdm ollama pyarrow

Note: you may need to restart the kernel to use updated packages.


## 1) Imports + configuration

In [ ]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timedelta, timezone
from typing import Dict, List, Any
import json
import re

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from pypdf import PdfReader
from tqdm.auto import tqdm

import faiss
from sentence_transformers import SentenceTransformer
from ollama import Client

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "fed_macro_mvp.ipynb").exists() and (PROJECT_DIR / "fed_macro_mvp").exists():
    PROJECT_DIR = PROJECT_DIR / "fed_macro_mvp"
DATA_DIR = PROJECT_DIR / "data"
RAW_PDF_DIR = DATA_DIR / "raw_pdfs"
PROCESSED_DIR = DATA_DIR / "processed"
INDEX_DIR = PROJECT_DIR / "index"
OUTPUT_DIR = PROJECT_DIR / "outputs"
for p in [RAW_PDF_DIR, PROCESSED_DIR, INDEX_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DAYS_BACK = 540
MAX_PDFS = 40
REQUEST_TIMEOUT = 20
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 8
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 180
OLLAMA_HOST = "http://127.0.0.1:11434"
# OLLAMA_MODEL = "mistral:7b-instruct"
OLLAMA_MODEL = "llama3:8b"

FOCUS_TOPICS = ["inflation", "unemployment", "labor market", "growth", "interest rates", "financial conditions", "credit"]
SEED_PAGES = [
    "https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm",
    "https://www.federalreserve.gov/monetarypolicy/fomcminutes.htm",
    "https://www.federalreserve.gov/monetarypolicy/mpr_default.htm",
    "https://www.federalreserve.gov/monetarypolicy/beigebookdefault.htm",
]

print("Project:", PROJECT_DIR)

/Users/atheeshkrishnan/AK/DEV/hawkdove/storied/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project: /Users/atheeshkrishnan/AK/DEV/hawkdove/fed_macro_mvp


## 2) Discover + download Federal Reserve PDFs

In [ ]:
def _normalize_pdf_url(href: str, base_url: str) -> str | None:
    href = (href or "").strip()
    if not href:
        return None
    if href.startswith("//"):
        href = "https:" + href
    elif href.startswith("/"):
        href = "https://www.federalreserve.gov" + href
    elif not href.lower().startswith("http"):
        href = requests.compat.urljoin(base_url, href)
    if ".pdf" not in href.lower() or "federalreserve.gov" not in href.lower():
        return None
    return href.split("#")[0]


def _extract_date_hint(s: str) -> str | None:
    m = re.search(r"(20\d{2})(\d{2})(\d{2})", s)
    if m:
        y, mo, d = m.groups()
        return f"{y}-{mo}-{d}"
    y = re.search(r"(20\d{2})", s)
    return f"{y.group(1)}-01-01" if y else None


def scrape_seed_pages(seed_pages: List[str]) -> pd.DataFrame:
    rows = []
    s = requests.Session()
    s.headers.update({"User-Agent": "Mozilla/5.0 (compatible; hawkdove-mvp/1.0)"})
    for page in seed_pages:
        try:
            r = s.get(page, timeout=REQUEST_TIMEOUT)
            if r.status_code != 200:
                continue
        except Exception:
            continue
        soup = BeautifulSoup(r.text, "html.parser")
        for a in soup.find_all("a"):
            pdf_url = _normalize_pdf_url(a.get("href"), page)
            if not pdf_url:
                continue
            title = " ".join(a.get_text(" ", strip=True).split()) or Path(pdf_url).name
            rows.append({"source": "seed_page", "source_page": page, "pdf_url": pdf_url, "title": title, "date_hint": _extract_date_hint(pdf_url)})
    if not rows:
        return pd.DataFrame(columns=["source", "source_page", "pdf_url", "title", "date_hint"])
    return pd.DataFrame(rows).drop_duplicates(subset=["pdf_url"]).reset_index(drop=True)


def probe_known_patterns(days_back: int = DAYS_BACK) -> pd.DataFrame:
    base = "https://www.federalreserve.gov/monetarypolicy/files/"
    s = requests.Session()
    s.headers.update({"User-Agent": "Mozilla/5.0 (compatible; hawkdove-mvp/1.0)"})
    rows = []
    today = datetime.now(timezone.utc).date()
    for offset in tqdm(range(days_back), desc="Probing patterns"):
        dt = today - timedelta(days=offset)
        ymd = dt.strftime("%Y%m%d")
        for fname in [f"fomcminutes{ymd}.pdf", f"monetary{ymd}a1.pdf"]:
            url = base + fname
            try:
                r = s.head(url, allow_redirects=True, timeout=8)
            except Exception:
                continue
            ctype = (r.headers.get("Content-Type") or "").lower()
            if r.status_code == 200 and ("pdf" in ctype or url.endswith(".pdf")):
                rows.append({"source": "known_pattern", "source_page": "monetarypolicy/files", "pdf_url": url, "title": fname, "date_hint": dt.isoformat()})
    if not rows:
        return pd.DataFrame(columns=["source", "source_page", "pdf_url", "title", "date_hint"])
    return pd.DataFrame(rows).drop_duplicates(subset=["pdf_url"]).reset_index(drop=True)


def build_catalog(seed_pages: List[str], days_back: int = DAYS_BACK) -> pd.DataFrame:
    a = scrape_seed_pages(seed_pages)
    b = probe_known_patterns(days_back)
    c = pd.concat([a, b], ignore_index=True).drop_duplicates(subset=["pdf_url"])
    if c.empty:
        return c
    c["parsed_date"] = pd.to_datetime(c["date_hint"], errors="coerce")
    return c.sort_values("parsed_date", ascending=False, na_position="last").reset_index(drop=True)


def download_catalog(catalog_df: pd.DataFrame, out_dir: Path, max_pdfs: int = MAX_PDFS) -> pd.DataFrame:
    if catalog_df.empty:
        return catalog_df
    s = requests.Session()
    s.headers.update({"User-Agent": "Mozilla/5.0 (compatible; hawkdove-mvp/1.0)"})
    rows = []
    for _, row in catalog_df.head(max_pdfs).iterrows():
        url = row["pdf_url"]
        fname = Path(url).name
        local = out_dir / fname
        if local.exists() and local.stat().st_size > 0:
            rows.append({**row.to_dict(), "local_path": str(local), "status": "exists"})
            continue
        try:
            r = s.get(url, timeout=REQUEST_TIMEOUT)
            if r.status_code == 200 and r.content:
                local.write_bytes(r.content)
                rows.append({**row.to_dict(), "local_path": str(local), "status": "downloaded"})
            else:
                rows.append({**row.to_dict(), "local_path": str(local), "status": f"http_{r.status_code}"})
        except Exception as e:
            rows.append({**row.to_dict(), "local_path": str(local), "status": f"error:{e}"})
    return pd.DataFrame(rows)


catalog_df = build_catalog(SEED_PAGES, DAYS_BACK)
print("Catalog candidates:", len(catalog_df))
if not catalog_df.empty:
    display(catalog_df.head(20))

download_df = download_catalog(catalog_df, RAW_PDF_DIR, MAX_PDFS)
print("Downloaded/available:", len(download_df))
if not download_df.empty:
    display(download_df[["title", "status", "local_path"]].head(20))

manifest_path = PROCESSED_DIR / "download_manifest.csv"
if not download_df.empty:
    download_df.to_csv(manifest_path, index=False)
    print("Saved:", manifest_path)


Probing patterns:   9%|▉         | 50/540 [00:06<01:03,  7.77it/s]

## 3) Parse + chunk + index

In [ ]:
def read_pdf_text(path: Path) -> str:
    try:
        reader = PdfReader(str(path))
    except Exception:
        return ""
    pages = []
    for p in reader.pages:
        try:
            pages.append(p.extract_text() or "")
        except Exception:
            pages.append("")
    text = "\n".join(pages)
    return re.sub(r"\s+", " ", text).strip()


def chunk_text(text: str, size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    if not text:
        return []
    out = []
    step = max(1, size - overlap)
    i = 0
    while i < len(text):
        j = min(len(text), i + size)
        out.append(text[i:j])
        if j == len(text):
            break
        i += step
    return out


def get_topic_flags(text: str) -> List[str]:
    t = text.lower()
    return [x for x in FOCUS_TOPICS if x in t]


def build_chunks(download_manifest: pd.DataFrame) -> pd.DataFrame:
    rows = []
    if download_manifest.empty:
        return pd.DataFrame()
    ok = download_manifest[download_manifest["status"].isin(["downloaded", "exists"])].copy()
    for _, row in tqdm(ok.iterrows(), total=len(ok), desc="Parsing PDFs"):
        f = Path(row["local_path"])
        if not f.exists():
            continue
        text = read_pdf_text(f)
        if len(text) < 200:
            continue
        parts = chunk_text(text)
        for idx, part in enumerate(parts):
            if len(part.strip()) < 80:
                continue
            rows.append({
                "chunk_id": f"{f.name}::chunk{idx:04d}",
                "doc_id": f.name,
                "pdf_url": row.get("pdf_url", ""),
                "date_hint": row.get("date_hint", ""),
                "chunk_index": idx,
                "text": part,
                "topic_flags": get_topic_flags(part),
                "text_len": len(part),
            })
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).drop_duplicates(subset=["chunk_id"]).reset_index(drop=True)


def norm_rows(x: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(x, axis=1, keepdims=True)
    n[n == 0] = 1.0
    return x / n


def save_table(df: pd.DataFrame, preferred_path: Path) -> Path:
    try:
        df.to_parquet(preferred_path, index=False)
        return preferred_path
    except Exception as e:
        fallback = preferred_path.with_suffix(".csv")
        df.to_csv(fallback, index=False)
        print(f"[warn] Parquet unavailable ({e}); saved CSV fallback: {fallback}")
        return fallback


def build_faiss_index(chunks_df: pd.DataFrame, embed_model_name: str = EMBED_MODEL_NAME) -> Dict[str, Any]:
    if chunks_df.empty:
        raise ValueError("No chunks. Run ingestion first.")
    model = SentenceTransformer(embed_model_name)
    vecs = model.encode(chunks_df["text"].tolist(), show_progress_bar=True, convert_to_numpy=True).astype("float32")
    vecs = norm_rows(vecs)
    dim = vecs.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(vecs)

    index_path = INDEX_DIR / "fed_chunks.index"
    meta_path = save_table(chunks_df, INDEX_DIR / "fed_chunks_meta.parquet")
    cfg_path = INDEX_DIR / "index_config.json"

    faiss.write_index(index, str(index_path))
    

    cfg = {
        "embed_model": embed_model_name,
        "dimension": dim,
        "rows": int(len(chunks_df)),
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "index_path": str(index_path),
        "meta_path": str(meta_path),
    }
    with open(cfg_path, "w") as f:
        json.dump(cfg, f, indent=2)
    return cfg


chunks_df = build_chunks(download_df)
print("Chunk rows:", len(chunks_df))
if not chunks_df.empty:
    display(chunks_df.head(5))
    chunks_path = save_table(chunks_df, PROCESSED_DIR / "chunks.parquet")
    print("Saved:", chunks_path)

if not chunks_df.empty:
    cfg = build_faiss_index(chunks_df, EMBED_MODEL_NAME)
    print(json.dumps(cfg, indent=2))


## 4) Retrieval + local LLM macro synthesis

In [ ]:
def load_bundle(index_dir: Path, embed_model_name: str = EMBED_MODEL_NAME):
    idx = faiss.read_index(str(index_dir / "fed_chunks.index"))
    meta_parquet = index_dir / "fed_chunks_meta.parquet"
    meta_csv = index_dir / "fed_chunks_meta.csv"
    if meta_parquet.exists():
        meta = pd.read_parquet(meta_parquet)
    elif meta_csv.exists():
        meta = pd.read_csv(meta_csv)
    else:
        raise FileNotFoundError("No index metadata found (expected fed_chunks_meta.parquet or fed_chunks_meta.csv)")
    em = SentenceTransformer(embed_model_name)
    return idx, meta, em


def retrieve(query: str, idx, meta_df: pd.DataFrame, emb_model, top_k: int = TOP_K) -> pd.DataFrame:
    q = emb_model.encode([query], convert_to_numpy=True).astype("float32")
    q = norm_rows(q)
    scores, ids = idx.search(q, top_k)
    rows = []
    for score, i in zip(scores[0], ids[0]):
        if i < 0 or i >= len(meta_df):
            continue
        r = meta_df.iloc[int(i)].to_dict()
        r["score"] = float(score)
        rows.append(r)
    return pd.DataFrame(rows)


def context_from_hits(hits_df: pd.DataFrame) -> str:
    blocks = []
    for _, r in hits_df.iterrows():
        t = str(r["text"]).replace("\n", " ").strip()
        blocks.append(f"[chunk_id={r['chunk_id']}; doc_id={r['doc_id']}; score={r['score']:.3f}]\n{t}")
    return "\n\n".join(blocks)




def check_ollama_ready(host: str = OLLAMA_HOST, model: str = OLLAMA_MODEL) -> None:
    try:
        r = requests.get(f"{host}/api/tags", timeout=3)
        r.raise_for_status()
        data = r.json()
    except Exception as e:
        raise RuntimeError(
            "Ollama is not reachable. Start Ollama first (e.g., `ollama serve` or open the Ollama app). "
            f"Host tried: {host}. Original error: {e}"
        )

    names = {m.get("name", "") for m in data.get("models", []) if isinstance(m, dict)}
    if model not in names:
        hint = ", ".join(sorted(names)) if names else "<no models found>"
        raise RuntimeError(
            f"Model '{model}' not found in Ollama. Available: {hint}. "
            f"Run: ollama pull {model} (or update OLLAMA_MODEL)."
        )


def generate_view(question: str, hits_df: pd.DataFrame, model: str = OLLAMA_MODEL) -> str:
    system = " ".join([
        "You are a U.S. macro research analyst focused on Federal Reserve communications.",
        "Use only provided context and be explicit about uncertainty.",
        "Return strict JSON with keys:",
        "executive_summary, outlook_horizon, inflation, unemployment, growth, policy_rates, risks, confidence, citations.",
        "For inflation/unemployment/growth/policy_rates/risks use object format with keys view and evidence.",
        "citations must be objects with chunk_id and why_relevant.",
        "Do not invent chunk IDs.",
    ])
    user = f"Question: {question}\n\nContext:\n{context_from_hits(hits_df)}"
    client = Client(host=OLLAMA_HOST)
    out = client.chat(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        options={"temperature": 0.1},
    )
    return out["message"]["content"]


def parse_json(text: str) -> Dict[str, Any] | None:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text).strip()
        text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, flags=re.S)
        if not m:
            return None
        try:
            return json.loads(m.group(0))
        except Exception:
            return None


index, meta_df, emb_model = load_bundle(INDEX_DIR, EMBED_MODEL_NAME)
query = "Based on the latest Federal Reserve communications, provide a 6-12 month view on inflation, unemployment, growth, and policy rates."
hits_df = retrieve(query, index, meta_df, emb_model, TOP_K)
print("Retrieved hits:", len(hits_df))
display(hits_df[["chunk_id", "doc_id", "date_hint", "score", "topic_flags"]].head(10))

check_ollama_ready(OLLAMA_HOST, OLLAMA_MODEL)
llm_text = generate_view(query, hits_df, OLLAMA_MODEL)
print(llm_text)

parsed = parse_json(llm_text)
if parsed is None:
    print("[check] JSON parse failed")
else:
    missing = [k for k in ["inflation", "unemployment", "growth", "policy_rates", "risks"] if k not in parsed]
    print("[check] Missing sections:", missing if missing else "None")
    valid_ids = set(hits_df["chunk_id"].tolist())
    cited_ids = []
    if isinstance(parsed.get("citations"), list):
        for c in parsed["citations"]:
            if isinstance(c, dict) and c.get("chunk_id"):
                cited_ids.append(c["chunk_id"])
    bad = sorted(set(cited_ids) - valid_ids)
    print("[check] Unknown cited IDs:", bad if bad else "None")


## 5) Save artifacts

In [ ]:
run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")

hits_path = OUTPUT_DIR / f"hits_{run_ts}.csv"
hits_df.to_csv(hits_path, index=False)
print("Saved:", hits_path)

txt_path = OUTPUT_DIR / f"macro_answer_{run_ts}.txt"
txt_path.write_text(llm_text)
print("Saved:", txt_path)

if parsed is not None:
    json_path = OUTPUT_DIR / f"macro_answer_{run_ts}.json"
    with open(json_path, "w") as f:
        json.dump(parsed, f, indent=2)
    print("Saved:", json_path)
